# Chapter 1 - Your First Agent in 15 Minutes

Build a minimal agent fast, then observe where it falls short.

## Learn
- What a thin configuration looks like
- How to bind a semantic view as a Cortex Analyst tool


In [ ]:
-- Build a deliberately thin semantic view
USE ROLE SYSADMIN;
USE WAREHOUSE SFE_SAM_SNOWMAN_WH;

CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML(
  'SNOWFLAKE_EXAMPLE.SEMANTIC_MODELS',
  $$
name: SV_SAM_DRIFT_NAIVE
description: Thin baseline semantic view for Drift walkthrough
tables:
  - name: SNOWFLAKE_EXAMPLE.SAM_DRIFT.CUSTOMER
  - name: SNOWFLAKE_EXAMPLE.SAM_DRIFT.INVOICE
  - name: SNOWFLAKE_EXAMPLE.SAM_DRIFT.INVOICE_LINE
  - name: SNOWFLAKE_EXAMPLE.SAM_DRIFT.TRACK
  - name: SNOWFLAKE_EXAMPLE.SAM_DRIFT.GENRE
verified_queries:
  - name: top_genres
    question: What genres does Drift sell most of?
    sql: |
      SELECT g.name AS genre_name, SUM(il.quantity) AS units
      FROM SNOWFLAKE_EXAMPLE.SAM_DRIFT.INVOICE_LINE il
      JOIN SNOWFLAKE_EXAMPLE.SAM_DRIFT.TRACK t ON il.track_id = t.track_id
      JOIN SNOWFLAKE_EXAMPLE.SAM_DRIFT.GENRE g ON t.genre_id = g.genre_id
      GROUP BY 1
      ORDER BY 2 DESC
      LIMIT 10
$$,
  FALSE
);


In [ ]:
-- Create the naive Sam agent
CREATE OR REPLACE AGENT SNOWFLAKE_EXAMPLE.SAM_DRIFT.SAM_THE_SNOWMAN
  COMMENT = 'DEMO: Sam naive Drift agent (Expires: 2026-05-22)'
  PROFILE = '{"display_name": "Sam-the-Snowman", "color": "blue"}'
  FROM SPECIFICATION
$$
models:
  orchestration: auto

instructions:
  system: |
    You are Sam, a data assistant for Drift Entertainment.

tools:
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: drift_naive_query
      description: Run basic analytics on Drift tables.
    tool_resources:
      semantic_view: SNOWFLAKE_EXAMPLE.SEMANTIC_MODELS.SV_SAM_DRIFT_NAIVE
      execution_environment:
        warehouse: SFE_SAM_SNOWMAN_WH
$$;


In [ ]:
-- Check
SELECT SNOWFLAKE.CORTEX.AGENT(
  'SNOWFLAKE_EXAMPLE.SAM_DRIFT.SAM_THE_SNOWMAN',
  PARSE_JSON($$
    {"messages":[{"role":"user","content":"What genres does Drift sell most of?"}]}
  $$)
) AS response;


## Reflect
This baseline is intentionally under-specified. It succeeds on straightforward prompts but lacks guardrails for ambiguous analytics.

## Next
Open [`ch02_break_it_on_purpose.ipynb`](ch02_break_it_on_purpose.ipynb).
